In [1]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\playdata2\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\playdata2\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\playdata2\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
sentences = [          
    'nice great best amazing',  # 긍정 문장 예시
    'stop lies',                # 부정/비판 문장 예시
    'pitiful nerd',             # 부정 문장 예시
    'excellent work',           # 긍정 문장 예시
    'supreme quality',          # 긍정 문장 예시
    'bad',                      # 부정 문장 예시
    'highly respectable'        # 긍정 문장 예시
]                               # 분류 모델에 넣을 입력 문장 리스트(list[str])
labels = [1, 0, 0, 1, 1, 0, 1]  # 각 문장에 대한 이진 라벨(1=긍정, 0=부정)

In [3]:
from nltk.tokenize import word_tokenize

tokenized_sentences = [word_tokenize(sent) for sent in sentences]
tokenized_sentences

[['nice', 'great', 'best', 'amazing'],
 ['stop', 'lies'],
 ['pitiful', 'nerd'],
 ['excellent', 'work'],
 ['supreme', 'quality'],
 ['bad'],
 ['highly', 'respectable']]

In [6]:
from collections import Counter

tokens = [token for sent in tokenized_sentences for token in sent]
word_counts = Counter(tokens)
print(word_counts)

word_to_index = {word: index + 2 for index, word in enumerate(tokens)}
word_to_index['<PAD>'] = 0
word_to_index['<UNK>'] = 1
word_to_index = dict(sorted(word_to_index.items(), key=lambda x: x[1]))
print(word_to_index)

vocab_size = len(word_to_index)
vocab_size

Counter({'nice': 1, 'great': 1, 'best': 1, 'amazing': 1, 'stop': 1, 'lies': 1, 'pitiful': 1, 'nerd': 1, 'excellent': 1, 'work': 1, 'supreme': 1, 'quality': 1, 'bad': 1, 'highly': 1, 'respectable': 1})
{'<PAD>': 0, '<UNK>': 1, 'nice': 2, 'great': 3, 'best': 4, 'amazing': 5, 'stop': 6, 'lies': 7, 'pitiful': 8, 'nerd': 9, 'excellent': 10, 'work': 11, 'supreme': 12, 'quality': 13, 'bad': 14, 'highly': 15, 'respectable': 16}


17

In [8]:
def texts_to_sequences(sentences, word_to_index):
    sequences = []

    for sent in sentences:
        sequence = []

        for token in sent:
            if token in word_to_index:
                sequence.append(word_to_index[token])
            else:
                sequence.append(word_to_index['<UNK>'])
        sequences.append(sequence)

    return sequences

sequences = texts_to_sequences(tokenized_sentences, word_to_index)
sequences

[[2, 3, 4, 5], [6, 7], [8, 9], [10, 11], [12, 13], [14], [15, 16]]

In [10]:
import numpy as np

def pad_sequences(sentences, maxlen):
    padded_sequences = np.zeros((len(sequences), maxlen), dtype=int)
    for index, seq in enumerate(sequences):
        padded_sequences[index, :len(seq)] = seq[:maxlen]
    return padded_sequences

padded_sequences = pad_sequences(sequences, maxlen=4)
padded_sequences

array([[ 2,  3,  4,  5],
       [ 6,  7,  0,  0],
       [ 8,  9,  0,  0],
       [10, 11,  0,  0],
       [12, 13,  0,  0],
       [14,  0,  0,  0],
       [15, 16,  0,  0]])

In [11]:
padded_sequences.shape

(7, 4)

In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

class SimpleNet(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, 1)

    def forward(self, x):
        embedded = self.embedding(x)
        # out: logit 값, h_n: (num_layers*directions, batch, hidden_size)
        out, h_n = self.rnn(embedded)
        out = self.out(h_n.squeeze(0))   # (batch, hidden_size) -> (batch, 1)
        return out

embedding_dim = 100
model = SimpleNet(vocab_size, embedding_dim, hidden_size=16)
model

SimpleNet(
  (embedding): Embedding(17, 100, padding_idx=0)
  (rnn): RNN(100, 16, batch_first=True)
  (out): Linear(in_features=16, out_features=1, bias=True)
)

In [15]:
from torchinfo import summary

summary(model)

Layer (type:depth-idx)                   Param #
SimpleNet                                --
├─Embedding: 1-1                         1,700
├─RNN: 1-2                               1,888
├─Linear: 1-3                            17
Total params: 3,605
Trainable params: 3,605
Non-trainable params: 0

In [16]:
import pandas as pd

wv = model.embedding.weight.data
print(wv.shape)

vocab = word_to_index.keys()
pd.DataFrame(wv, index=vocab)

torch.Size([17, 100])


,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
<PAD>,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
<UNK>,-1.532296,-0.047054,0.192900,-0.215433,0.292076,-0.855446,-1.587114,0.517788,0.508944,1.559252,...,-0.873326,-0.426199,-0.545862,0.478442,1.568153,-0.597651,0.356423,0.155171,0.104144,0.425359
nice,-0.791466,1.042925,0.707471,-1.596295,-0.478177,0.351474,-1.426117,1.328005,-0.544260,1.619726,...,-1.162552,-0.437493,2.798892,0.803716,0.759993,0.520603,1.046477,0.372067,-0.953278,-0.857929
great,-0.802728,2.186314,0.326941,0.086670,-0.362339,-0.551848,0.292478,-0.908567,-0.589190,-0.328993,...,1.561370,0.816675,-0.795487,-0.336552,0.461962,-1.241810,-0.487442,1.141592,0.916974,-0.680340
best,-0.602169,0.020619,0.183081,0.867959,0.188726,-1.764698,0.206935,1.095213,1.881546,-0.104011,...,-1.214042,-1.086225,-1.169131,0.160394,-0.896053,-0.188697,-0.634172,-2.707866,0.842657,1.021467
amazing,-0.488972,-0.569514,-0.454523,0.825377,0.238835,-0.095774,-0.532069,-0.570310,-0.096213,-1.077382,...,-1.282428,-0.064538,1.524943,0.315079,-0.196659,0.116959,-0.020030,-0.373040,0.636681,-0.098134
stop,0.389420,-0.288761,1.600015,1.478264,-0.513109,0.293383,-1.826037,-0.224863,-0.742802,0.692731,...,-2.193576,1.152639,-2.217465,-0.450518,-1.042366,-0.544405,-0.574070,-0.580668,0.394284,-1.213821
lies,-0.749662,-1.172501,-0.477055,0.795381,-1.152480,-0.637103,-0.044801,-0.853359,-0.383779,-0.094142,...,-0.874396,0.670120,1.173817,2.384690,1.011771,0.646197,0.363759,1.610150,-1.205093,-0.050036
pitiful,0.283898,-0.165400,1.433685,1.071283,0.191908,0.607466,-0.760558,-1.769915,0.254453,-0.684930,...,-0.042748,1.873285,1.029849,0.430444,1.091727,-1.420720,-0.513713,-0.873070,-0.350664,-0.820339
nerd,-0.143462,-1.552643,-3.111761,-0.627048,-1.906358,-0.025553,-1.309372,-0.171639,-0.717444,-1.942175,...,1.235968,0.682064,-1.744367,0.463351,-0.509631,-0.255897,-1.300219,0.136173,1.429269,1.364069


In [17]:
X = torch.tensor(padded_sequences, dtype=torch.long)
y = torch.tensor(labels, dtype=torch.float).unsqueeze(1)

dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

In [18]:
for epoch in range(20):
    epoch_loss = 0

    for x_batch, y_batch in dataloader:
        optimizer.zero_grad()
        output = model(x_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f'Epoch {epoch+1} Loss: {epoch_loss / len(dataloader):.5f}')

Epoch 1 Loss: 0.73192
Epoch 2 Loss: 0.58388
Epoch 3 Loss: 0.46194
Epoch 4 Loss: 0.41972
Epoch 5 Loss: 0.30597
Epoch 6 Loss: 0.24392
Epoch 7 Loss: 0.18911
Epoch 8 Loss: 0.12345
Epoch 9 Loss: 0.08494
Epoch 10 Loss: 0.06230
Epoch 11 Loss: 0.04869
Epoch 12 Loss: 0.03879
Epoch 13 Loss: 0.03231
Epoch 14 Loss: 0.02696
Epoch 15 Loss: 0.02382
Epoch 16 Loss: 0.02028
Epoch 17 Loss: 0.01810
Epoch 18 Loss: 0.01606
Epoch 19 Loss: 0.01506
Epoch 20 Loss: 0.01408


In [22]:
model.eval()

with torch.no_grad():
    output = model(X)
    prob = torch.sigmoid(output)
    pred = (prob >= 0.5).int()

print(labels)
print(pred.squeeze().detach().numpy())

[1, 0, 0, 1, 1, 0, 1]
[1 0 0 1 1 0 1]


In [23]:
from gensim.models import KeyedVectors

model_wv = KeyedVectors.load_word2vec_format('GoogleNews-vectors-negative300.bin.gz', binary=True)
model_wv.vectors.shape

(3000000, 300)

In [25]:
print(len(word_to_index))

embedding_matrix = np.zeros((len(word_to_index), model_wv.vectors.shape[1]))
embedding_matrix.shape

17


(17, 300)

In [31]:
def get_word_embedding(word):
    if word in model_wv:
        return model_wv[word]
    else:
        return None

print(get_word_embedding('nerd'))

[ 2.65625000e-01 -2.07031250e-01 -2.66113281e-02  4.19921875e-01
 -2.08984375e-01  3.90625000e-01  1.64062500e-01  6.39648438e-02
  1.49414062e-01 -1.77001953e-02  2.41699219e-02 -1.48437500e-01
  2.83203125e-01 -2.26562500e-01 -1.61132812e-01 -2.85644531e-02
 -3.56445312e-02 -5.46875000e-02  1.45507812e-01 -1.12304688e-01
 -1.88476562e-01  9.86328125e-02  6.64062500e-02  4.07714844e-02
  3.01513672e-02 -2.00195312e-01 -6.12792969e-02  1.58203125e-01
  2.99072266e-02 -9.86328125e-02  2.48046875e-01 -1.26953125e-02
  3.17382812e-02  2.98828125e-01  5.37109375e-03 -9.71679688e-02
  3.10546875e-01 -1.42578125e-01  2.26562500e-01  2.73437500e-01
  9.08203125e-02 -5.05371094e-02  1.70898438e-01  2.51953125e-01
  6.80541992e-03 -1.66015625e-01 -1.51367188e-01  8.30078125e-03
  2.57568359e-02 -4.15039062e-02 -6.01562500e-01  1.06445312e-01
 -1.55639648e-02 -1.52343750e-01  3.61328125e-02  2.41699219e-02
 -2.17285156e-02 -4.06250000e-01 -1.54296875e-01  2.24609375e-02
 -8.83789062e-02  1.07421

In [33]:
for word, index in word_to_index.items():
    if index >= 2:
        emb = get_word_embedding(word)
        if emb is not None:
            embedding_matrix[index] = emb
            

In [34]:
pd.DataFrame(embedding_matrix, index=word_to_index.keys())

,0,1,2,3,4,5,6,7,8,9,...,290,291,292,293,294,295,296,297,298,299
<PAD>,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
<UNK>,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
nice,0.158203,0.105957,-0.189453,0.386719,0.083496,-0.267578,0.083496,0.113281,-0.104004,0.178711,...,-0.085449,0.189453,-0.146484,0.134766,-0.040771,0.032715,0.089355,-0.267578,0.008362,-0.213867
great,0.071777,0.208008,-0.028442,0.178711,0.132812,-0.099609,0.096191,-0.116699,-0.008545,0.148438,...,-0.011475,0.064453,-0.289062,-0.048096,-0.199219,-0.071289,0.064453,-0.167969,-0.020874,-0.142578
best,-0.126953,0.021973,0.287109,0.153320,0.127930,0.032715,-0.115723,-0.029541,0.153320,0.011292,...,0.006439,-0.033936,-0.166016,-0.016846,-0.048584,-0.022827,-0.152344,-0.101562,-0.090332,0.088379
amazing,0.073730,0.004059,-0.135742,0.022095,0.180664,-0.046631,0.224609,-0.229492,-0.040039,0.225586,...,0.018433,-0.021240,-0.250000,-0.020142,-0.310547,-0.207031,-0.006317,-0.141602,-0.150391,-0.137695
stop,-0.057861,0.013184,0.115234,0.069824,-0.306641,-0.044678,0.048584,0.152344,0.073242,-0.100098,...,0.100098,0.171875,-0.113281,0.064453,-0.115723,0.048096,-0.004822,0.086426,0.029907,0.007812
lies,0.149414,-0.012817,0.328125,0.025513,0.017334,0.190430,0.188477,-0.143555,-0.090820,0.206055,...,-0.308594,0.183594,-0.202148,0.031494,-0.164062,-0.201172,0.080078,-0.105469,0.149414,0.157227
pitiful,0.269531,0.253906,-0.020996,0.060303,-0.010925,0.217773,0.139648,-0.057617,0.312500,0.253906,...,-0.063477,0.132812,-0.094238,0.089355,-0.065430,-0.016235,-0.107910,-0.072266,-0.094238,0.028809
nerd,0.265625,-0.207031,-0.026611,0.419922,-0.208984,0.390625,0.164062,0.063965,0.149414,-0.017700,...,0.215820,0.125000,-0.227539,-0.310547,-0.112793,-0.096680,0.255859,0.124023,-0.030273,0.082031


In [37]:
a = torch.tensor([1., 2., 3.], requires_grad=False)
print(a.requires_grad)
b = torch.tensor([1., 2., 3.], requires_grad=True)
print(b.requires_grad)

False
True


In [38]:
class SimpleNet(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )

        self.embedding.weight = nn.Parameter(torch.tensor(embedding_matrix, dtype=torch.float))
        # self.embedding.weight.requires_grad = False

        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, 1)

    def forward(self, x):
        embedded = self.embedding(x)
        # out: logit 값, h_n: (num_layers*directions, batch, hidden_size)
        out, h_n = self.rnn(embedded)
        out = self.out(h_n.squeeze(0))   # (batch, hidden_size) -> (batch, 1)
        return out

embedding_dim = model_wv.vectors.shape[1]
model = SimpleNet(vocab_size, embedding_dim, hidden_size=16)
model

SimpleNet(
  (embedding): Embedding(17, 300, padding_idx=0)
  (rnn): RNN(300, 16, batch_first=True)
  (out): Linear(in_features=16, out_features=1, bias=True)
)

In [39]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

In [41]:
X = torch.tensor(padded_sequences, dtype=torch.long)
y = torch.tensor(labels, dtype=torch.float).unsqueeze(1)

dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

In [42]:
for epoch in range(20):
    epoch_loss = 0

    for x_batch, y_batch in dataloader:
        optimizer.zero_grad()
        output = model(x_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f'Epoch {epoch+1} Loss: {epoch_loss / len(dataloader):.5f}')

Epoch 1 Loss: 0.70114
Epoch 2 Loss: 0.53860
Epoch 3 Loss: 0.42403
Epoch 4 Loss: 0.31470
Epoch 5 Loss: 0.21625
Epoch 6 Loss: 0.14605
Epoch 7 Loss: 0.10191
Epoch 8 Loss: 0.07271
Epoch 9 Loss: 0.05389
Epoch 10 Loss: 0.04162
Epoch 11 Loss: 0.03270
Epoch 12 Loss: 0.02692
Epoch 13 Loss: 0.02169
Epoch 14 Loss: 0.02017
Epoch 15 Loss: 0.01697
Epoch 16 Loss: 0.01504
Epoch 17 Loss: 0.01352
Epoch 18 Loss: 0.01223
Epoch 19 Loss: 0.01080
Epoch 20 Loss: 0.01033


In [45]:
model.eval()

with torch.no_grad():
    output = model(X)
    prob = torch.sigmoid(output)
    pred = (prob >= 0.5).int()

print(labels)
print(pred.squeeze().detach().numpy())

[1, 0, 0, 1, 1, 0, 1]
[1 0 0 1 1 0 1]
